# Bibliotecas

In [32]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

import ipywidgets as widgets # código feito por IA
from IPython.display import display, clear_output # código feito por IA

pd.set_option('display.max_rows', None)

# Carregamento do JSON

In [104]:
with open("data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])
print(df.shape)  # deve dar (6674, 7)

(6674, 7)


# Visualização dos dados

In [ ]:
#Código feito com IA para auxiliar na visualização dos dados e identificação de problemas do JSON, para posteriormente podermos corrigir com código.
bairros_ordenados = sorted(df["bairro"].unique())

# Mapeamento com índice numérico (1 a 94) para facilitar a seleção
opcoes_bairros = [(f"{i+1} - {bairro}", bairro) for i, bairro in enumerate(bairros_ordenados)]

# 3. Criação dos Widgets Interativos
dropdown_bairro = widgets.Dropdown(
    options=opcoes_bairros,
    value=bairros_ordenados[17],
    description="Bairro:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

slider_niveis = widgets.IntSlider(
    value=71,
    min=1,
    max=df["nivel_m"].nunique(),
    step=1,
    description="Qtd. Níveis:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

output_area = widgets.Output()

# 4. Função de atualização dinâmica dos resultados
def atualizar_exibicao(change=None):
    bairro_selecionado = dropdown_bairro.value
    qtd_niveis = slider_niveis.value
    
    # Filtrar o DataFrame pelo bairro selecionado e ordenar pelo nível do rio (m)
    df_bairro = df[df["bairro"] == bairro_selecionado].sort_values(by="nivel_m")
    resultado = df_bairro.head(qtd_niveis)
    
    with output_area:
        clear_output(wait=True)
        print(f"Bairro selecionado: {bairro_selecionado}")
        print(f"Exibindo os primeiros {len(resultado)} níveis de inundação:")
        display(resultado)

# Vincular os eventos dos widgets à função de atualização
dropdown_bairro.observe(atualizar_exibicao, names='value')
slider_niveis.observe(atualizar_exibicao, names='value')

# 5. Renderizar a interface interativa no Notebook
interface = widgets.VBox([
    widgets.HTML(value="### Painel Interativo de Análise de Inundação"),
    dropdown_bairro,
    slider_niveis,
    widgets.HTML(value="<hr/>"),
    output_area
])

display(interface)
atualizar_exibicao()

# IDENTIFICAÇÃO DE ERROS

In [ ]:

# 1. Garantir a ordenação por bairro e cota
df = df.sort_values(['bairro', 'nivel_m']).reset_index(drop=True)

# Guardar cópia dos dados brutos do JSON
df['buildings_raw'] = df['affected_buildings']
df['pop_raw'] = df['affected_population']

# 2. Função de Detecção e Classificação com Tratamento de Outliers Iniciais
def classificar_anomalias_bairro(df_bairro, limiar_pct=0.005):
    b = df_bairro['affected_buildings'].values.astype(float).copy()
    n = len(b)
    tipo_anomalia = ['OK'] * n
    is_outlier = [False] * n

    # -------------------------------------------------------------
    # PASSO 0: Identificar Outliers Iniciais (Bloco de erro no início)
    # Ex: Espírito Santo com 637/633 nas cotas 3.0m-3.4m e ~30 nas cotas seguintes
    # -------------------------------------------------------------
    if n >= 15:
        mediana_corpo = np.median(b[5:20]) # Mediana das cotas intermediárias
        for i in range(min(5, n)):
            # Se o valor inicial for 5x maior que a mediana do corpo do bairro
            if b[i] > (mediana_corpo * 5) and b[i] > 100 and mediana_corpo < 100:
                is_outlier[i] = True
                tipo_anomalia[i] = 'OUTLIER_VIA_MEDIANA'

    # -------------------------------------------------------------
    # PASSO A: Identificar Outliers Pontuais no meio da série (Spikes)
    # -------------------------------------------------------------
    for i in range(n):
        if is_outlier[i]:
            continue
        prev_val = b[i-1] if i > 0 else b[i]
        next_val = b[i+1] if i < n-1 else b[i]
        
        if b[i] > (prev_val + 50) and b[i] > (next_val + 50) and prev_val < 500:
            is_outlier[i] = True
            tipo_anomalia[i] = 'OUTLIER_VIA_MEDIANA'

    # -------------------------------------------------------------
    # PASSO B: Avaliar Quedas/Zeros por Variação Percentual (Sem Outliers)
    # -------------------------------------------------------------
    pico_acumulado = 0
    for i in range(n):
        if is_outlier[i]:
            continue
            
        val = b[i]
        
        # Só analisa se houver uma queda real em relação ao pico prévio
        if val < (pico_acumulado - 1) and pico_acumulado > 0:
            # Buscar o próximo valor válido à frente
            proximo_valido = val
            for j in range(i + 1, n):
                if not is_outlier[j] and b[j] > val:
                    proximo_valido = b[j]
                    break
            
            # Variação relativa do valor posterior em relação ao pico antigo
            variacao_relativa = (proximo_valido - pico_acumulado) / pico_acumulado
            
            if variacao_relativa > limiar_pct:
                tipo_anomalia[i] = 'BURACO_VIA_INTERPOLACAO'
            else:
                tipo_anomalia[i] = 'RUIDO_VIA_ESTABILIZACAO'
        else:
            pico_acumulado = max(pico_acumulado, val)
            
    return pd.Series(tipo_anomalia, index=df_bairro.index)

# Aplica a classificação corrigida no DataFrame
df['tipo_anomalia'] = df.groupby('bairro', group_keys=False).apply(classificar_anomalias_bairro)
df['tem_anomalia'] = df['tipo_anomalia'] != 'OK'

print("Célula 1 atualizada: Outliers iniciais de bloco (como no Espírito Santo) capturados!")

✅ Célula 1 atualizada: Outliers iniciais de bloco (como no Espírito Santo) capturados!


# VISUALIZAÇÃO DOS ERROS

In [106]:
# Configurar Pandas para exibir todas as linhas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f" Total de cotas anômalas no dataset: {df['tem_anomalia'].sum()}\n")

# 1. Resumo por tipo de erro e bairro
resumo_tipos = (
    df[df['tem_anomalia']]
    .groupby(['bairro', 'tipo_anomalia'])
    .size()
    .reset_index(name='cotas_afetadas')
    .sort_values('cotas_afetadas', ascending=False)
)
print("--- 1. Diagnóstico Estatístico das Anomalias por Bairro ---")
display(resumo_tipos)

# 2. Exibição Completa das Linhas Afetadas (Dados Brutos)
tabela_auditoria = df[df['tem_anomalia']][
    ['bairro', 'nivel_m', 'buildings_raw', 'pop_raw', 'tipo_anomalia']
].rename(columns={
    'buildings_raw': 'Predios_Brutos',
    'pop_raw': 'Pop_Bruta'
})

print("\n--- 2. Tabela de Auditoria Bruta (Antes da Correção) ---")
display(tabela_auditoria)

 Total de cotas anômalas no dataset: 75

--- 1. Diagnóstico Estatístico das Anomalias por Bairro ---


,bairro,tipo_anomalia,cotas_afetadas
0,Arquipélago,RUIDO_VIA_ESTABILIZACAO,65
2,Espírito Santo,OUTLIER_VIA_MEDIANA,6
1,Centro Histórico,BURACO_VIA_INTERPOLACAO,2
3,Lageado,BURACO_VIA_INTERPOLACAO,1
4,Rubem Berta,BURACO_VIA_INTERPOLACAO,1



--- 2. Tabela de Auditoria Bruta (Antes da Correção) ---


,bairro,nivel_m,Predios_Brutos,Pop_Bruta,tipo_anomalia
214,Arquipélago,3.1,4680,6411,RUIDO_VIA_ESTABILIZACAO
215,Arquipélago,3.2,4680,6411,RUIDO_VIA_ESTABILIZACAO
216,Arquipélago,3.3,4680,6411,RUIDO_VIA_ESTABILIZACAO
217,Arquipélago,3.4,4680,6411,RUIDO_VIA_ESTABILIZACAO
219,Arquipélago,3.6,4680,6411,RUIDO_VIA_ESTABILIZACAO
220,Arquipélago,3.7,4680,6411,RUIDO_VIA_ESTABILIZACAO
221,Arquipélago,3.8,4680,6411,RUIDO_VIA_ESTABILIZACAO
222,Arquipélago,3.9,4680,6411,RUIDO_VIA_ESTABILIZACAO
224,Arquipélago,4.1,4674,6411,RUIDO_VIA_ESTABILIZACAO
225,Arquipélago,4.2,4680,6411,RUIDO_VIA_ESTABILIZACAO


# CORREÇÃO DOS ERROS 

In [ ]:

# 1. Criar uma cópia explícita e independente para a base limpa
df_limpo = df.copy()

def corrigir_bairro_adaptativo(df_bairro):
    b = df_bairro['buildings_raw'].astype(float).copy()
    p = df_bairro['pop_raw'].astype(float).copy()
    tipos = df_bairro['tipo_anomalia'].values
    n = len(b)
    
    # PASSO 1: Marcar OUTLIERS e BURACOS como NaN
    # (Tanto o 2101 quanto o bloco inicial 637/633 viram NaN para serem eliminados)
    for i in range(n):
        if tipos[i] in ['OUTLIER_VIA_MEDIANA', 'BURACO_VIA_INTERPOLACAO']:
            b.iloc[i] = np.nan
            p.iloc[i] = np.nan
            
    # PASSO 2: Interpolação Linear / Preenchimento dos NaNs com os vizinhos VÁLIDOS
    # bfill() trata o caso de o outlier estar na primeira linha (cotas 3.0m a 3.4m)
    b = b.interpolate(method='linear').bfill().ffill()
    p = p.interpolate(method='linear').bfill().ffill()
    
    # PASSO 3: RUÍDOS EM PLATÔ (Garantir a monotonicidade física acumulada)
    b = np.maximum.accumulate(b.values)
    p = np.maximum.accumulate(p.values)
    
    return pd.DataFrame({
        'affected_buildings': np.round(b).astype(int),
        'affected_population': np.round(p).astype(int)
    }, index=df_bairro.index)

# 2. Aplicar as correções EXCLUSIVAMENTE no df_limpo
df_corrigido = df_limpo.groupby('bairro', group_keys=False).apply(corrigir_bairro_adaptativo)

df_limpo['affected_buildings'] = df_corrigido['affected_buildings']
df_limpo['affected_population'] = df_corrigido['affected_population']
df_limpo['percentage'] = (df_limpo['affected_buildings'] / df_limpo['total_buildings']) * 100

print("'df_limpo' atualizado: Outliers do Espírito Santo corrigidos para a baseline de ~25 prédios!")

✅ 'df_limpo' atualizado: Outliers do Espírito Santo corrigidos para a baseline de ~25 prédios!


# COMPARAÇÃO 

In [ ]:
# Cruzando os dados brutos de 'df' com os dados corrigidos de 'df_limpo'
comparativo_final = pd.DataFrame({
    'bairro': df.loc[df['tem_anomalia'], 'bairro'],
    'nivel_m': df.loc[df['tem_anomalia'], 'nivel_m'],
    'Estratégia_Aplicada': df.loc[df['tem_anomalia'], 'tipo_anomalia'],
    'Predios_Bruto (JSON)': df.loc[df['tem_anomalia'], 'buildings_raw'],
    'Predios_Corrigido (df_limpo)': df_limpo.loc[df['tem_anomalia'], 'affected_buildings']
})

print(f"Total de registros auditados: {len(comparativo_final)}\n")
print("--- Comparativo de Resultados (df Bruto vs. df_limpo) ---")
display(comparativo_final)

📌 Total de registros auditados: 75

--- Comparativo de Resultados (df Bruto vs. df_limpo) ---


,bairro,nivel_m,Estratégia_Aplicada,Predios_Bruto (JSON),Predios_Corrigido (df_limpo)
214,Arquipélago,3.1,RUIDO_VIA_ESTABILIZACAO,4680,4685
215,Arquipélago,3.2,RUIDO_VIA_ESTABILIZACAO,4680,4685
216,Arquipélago,3.3,RUIDO_VIA_ESTABILIZACAO,4680,4685
217,Arquipélago,3.4,RUIDO_VIA_ESTABILIZACAO,4680,4685
219,Arquipélago,3.6,RUIDO_VIA_ESTABILIZACAO,4680,4685
220,Arquipélago,3.7,RUIDO_VIA_ESTABILIZACAO,4680,4685
221,Arquipélago,3.8,RUIDO_VIA_ESTABILIZACAO,4680,4685
222,Arquipélago,3.9,RUIDO_VIA_ESTABILIZACAO,4680,4685
224,Arquipélago,4.1,RUIDO_VIA_ESTABILIZACAO,4674,4685
225,Arquipélago,4.2,RUIDO_VIA_ESTABILIZACAO,4680,4685
